# Stress Prediction — 개선 실험 및 제출 파일 생성

기존 노트북의 구조를 유지하면서 아래 항목을 안전하게 비교합니다.

1. 예측값 소수 둘째 자리 반올림 제거
2. 최근접 이웃 오버라이드 사용 여부와 임계값 비교
3. Pair 모델의 이웃 수 `k=1, 3` 비교
4. ExtraTrees의 `max_features`, `min_samples_leaf`, `criterion` 비교
5. 파생변수 추가

> 먼저 CV 비교 셀을 실행하고, 결과가 가장 좋은 설정을 `FINAL_CONFIG`에 반영한 뒤 제출 파일을 생성하세요.


In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass, replace
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
WINSORIZE_COLUMNS = [
    "mean_working", "diastolic_blood_pressure", "cholesterol", "glucose"
]

# 로컬 환경에 맞게 필요 시 수정하세요.
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


@dataclass(frozen=True)
class ModelConfig:
    name: str
    pair_weight: float = 0.24
    pair_k: int = 1
    round_decimals: int | None = None
    use_override: bool = True
    override_threshold: float = 0.20
    max_features: int | float | str = 1
    min_samples_leaf: int = 1
    criterion: str = "squared_error"
    use_feature_copies: bool = True
    include_gender: bool = False


BASELINE_CONFIG = ModelConfig(
    name="baseline_original",
    pair_weight=0.24,
    pair_k=1,
    round_decimals=2,
    use_override=True,
    override_threshold=0.20,
    max_features=1,
    min_samples_leaf=1,
    criterion="squared_error",
    use_feature_copies=True,
    include_gender=False,
)


In [2]:
def compute_iqr_bounds(train_frame: pd.DataFrame, columns: list[str]):
    bounds = {}
    for col in columns:
        if col not in train_frame.columns:
            continue
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame: pd.DataFrame, bounds):
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    return a / (b.abs() + 1e-6)


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = safe_divide(result["weight"], height_m.pow(2))
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = safe_divide(sbp, dbp)
    result["cholesterol_glucose_ratio"] = safe_divide(
        result["cholesterol"], result["glucose"]
    )

    # 추가 파생변수
    result["weight_height_ratio"] = safe_divide(result["weight"], result["height"])
    result["bmi_squared"] = result["bmi"].pow(2)
    result["bp_sum"] = sbp + dbp
    result["bp_product"] = sbp * dbp
    result["cholesterol_glucose_product"] = result["cholesterol"] * result["glucose"]
    result["glucose_bmi"] = result["glucose"] * result["bmi"]
    result["cholesterol_bmi"] = result["cholesterol"] * result["bmi"]
    result["working_bmi"] = result["mean_working"] * result["bmi"]
    result["working_glucose"] = result["mean_working"] * result["glucose"]
    result["working_cholesterol"] = result["mean_working"] * result["cholesterol"]

    for col in ["mean_working", "cholesterol", "glucose", "weight", "bmi"]:
        result[f"{col}_log1p"] = np.log1p(result[col].clip(lower=0))

    result["missing_count"] = frame.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()

    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def apply_prediction_rounding(pred: np.ndarray, decimals: int | None) -> np.ndarray:
    if decimals is not None:
        pred = np.round(pred, decimals)
    return np.clip(pred, 0.0, 1.0)


In [3]:
def tree_prediction(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    config: ModelConfig,
) -> np.ndarray:
    train_used = train_features.copy()
    val_used = val_features.copy()

    if not config.include_gender:
        train_used = train_used.drop(columns=["gender"], errors="ignore")
        val_used = val_used.drop(columns=["gender"], errors="ignore")

    if config.use_feature_copies:
        train_used = add_feature_copies(train_used)
        val_used = add_feature_copies(val_used)

    preprocessor = make_preprocessor(train_used)
    train_matrix = preprocessor.fit_transform(train_used)
    val_matrix = preprocessor.transform(val_used)

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=config.min_samples_leaf,
        max_features=config.max_features,
        criterion=config.criterion,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)

    tree_predictions = np.column_stack([
        estimator.predict(val_matrix) for estimator in model.estimators_
    ])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    k: int = 1,
) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []

    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )

        if k == 1:
            nearest_idx = np.argmin(distance, axis=1)
            pair_target = target_values[nearest_idx]
        else:
            nearest_indices = np.argpartition(distance, kth=k - 1, axis=1)[:, :k]
            pair_target = np.median(target_values[nearest_indices], axis=1)

        neighbor_targets.append(pair_target)

    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    medians = train_raw[numeric_cols].median()
    train_numeric = train_raw[numeric_cols].fillna(medians)
    val_numeric = val_raw[numeric_cols].fillna(medians)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)

    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def combined_prediction(
    train_raw_fold: pd.DataFrame,
    val_raw_fold: pd.DataFrame,
    target: pd.Series,
    numeric_cols: list[str],
    config: ModelConfig,
) -> np.ndarray:
    bounds = compute_iqr_bounds(train_raw_fold, WINSORIZE_COLUMNS)
    train_winsorized = apply_winsorize(train_raw_fold, bounds)
    val_winsorized = apply_winsorize(val_raw_fold, bounds)

    train_feat = add_features(train_winsorized)
    val_feat = add_features(val_winsorized)

    tree_pred = tree_prediction(train_feat, val_feat, target, config)
    pair_pred = pair_prediction(train_feat, val_feat, target, k=config.pair_k)

    base_pred = (
        (1.0 - config.pair_weight) * tree_pred
        + config.pair_weight * pair_pred
    )
    result = apply_prediction_rounding(base_pred, config.round_decimals)

    if config.use_override:
        nearest_dist, nearest_target = nearest_neighbor_lookup(
            train_raw_fold, val_raw_fold, target, numeric_cols
        )
        override_mask = nearest_dist < config.override_threshold
        override_values = apply_prediction_rounding(
            nearest_target, config.round_decimals
        )
        result[override_mask] = override_values[override_mask]

    return np.clip(result, 0.0, 1.0)


## 데이터 로드 및 후보 설정

첫 실행에서는 후보 수를 너무 늘리지 않는 것이 좋습니다. 아래 후보들은 기존 모델에서 한두 요소만 바꾼 버전이라 원인을 비교하기 쉽습니다.


In [4]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
y = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()
cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)

CANDIDATE_CONFIGS = [
    BASELINE_CONFIG,
    replace(
        BASELINE_CONFIG,
        name="no_rounding",
        round_decimals=None,
    ),
    replace(
        BASELINE_CONFIG,
        name="no_rounding_no_override",
        round_decimals=None,
        use_override=False,
    ),
    replace(
        BASELINE_CONFIG,
        name="no_rounding_override_010",
        round_decimals=None,
        override_threshold=0.10,
    ),
    replace(
        BASELINE_CONFIG,
        name="pair_k3_no_rounding",
        pair_k=3,
        round_decimals=None,
    ),
    replace(
        BASELINE_CONFIG,
        name="extra_trees_mf05_leaf2",
        round_decimals=None,
        max_features=0.5,
        min_samples_leaf=2,
        use_feature_copies=False,
    ),
    replace(
        BASELINE_CONFIG,
        name="extra_trees_absolute",
        round_decimals=None,
        criterion="absolute_error",
        max_features=0.5,
        min_samples_leaf=2,
        use_feature_copies=False,
    ),
]

print("Train shape:", train_raw.shape)
print("후보 수:", len(CANDIDATE_CONFIGS))


Train shape: (3000, 18)
후보 수: 7


## 5-Fold CV 비교

`absolute_error` 후보는 다른 후보보다 학습 시간이 오래 걸릴 수 있습니다. 결과표에서 **CV MAE가 가장 낮은 설정**을 확인하세요.


In [5]:
def evaluate_config(config: ModelConfig) -> tuple[float, np.ndarray]:
    oof = np.zeros(len(y), dtype=float)

    for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
        train_fold = raw_input.iloc[train_idx].reset_index(drop=True)
        val_fold = raw_input.iloc[val_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)

        oof[val_idx] = combined_prediction(
            train_fold,
            val_fold,
            y_train,
            numeric_cols,
            config,
        )
        print(f"[{config.name}] Fold {fold} 완료")

    return mean_absolute_error(y, oof), oof


cv_rows = []
oof_by_name = {}

for config in CANDIDATE_CONFIGS:
    mae, oof_pred = evaluate_config(config)
    oof_by_name[config.name] = oof_pred
    cv_rows.append({
        "name": config.name,
        "cv_mae": mae,
        "pair_weight": config.pair_weight,
        "pair_k": config.pair_k,
        "round_decimals": config.round_decimals,
        "use_override": config.use_override,
        "override_threshold": config.override_threshold,
        "max_features": config.max_features,
        "min_samples_leaf": config.min_samples_leaf,
        "criterion": config.criterion,
        "use_feature_copies": config.use_feature_copies,
    })
    print(f"{config.name}: CV MAE={mae:.8f}\n")

cv_results = pd.DataFrame(cv_rows).sort_values("cv_mae").reset_index(drop=True)
display(cv_results)

BEST_NAME = cv_results.loc[0, "name"]
BEST_CONFIG = next(config for config in CANDIDATE_CONFIGS if config.name == BEST_NAME)
print("CV 기준 최적 설정:", BEST_CONFIG)


[baseline_original] Fold 1 완료
[baseline_original] Fold 2 완료
[baseline_original] Fold 3 완료
[baseline_original] Fold 4 완료
[baseline_original] Fold 5 완료
baseline_original: CV MAE=0.14926000

[no_rounding] Fold 1 완료
[no_rounding] Fold 2 완료
[no_rounding] Fold 3 완료
[no_rounding] Fold 4 완료
[no_rounding] Fold 5 완료
no_rounding: CV MAE=0.14928563

[no_rounding_no_override] Fold 1 완료
[no_rounding_no_override] Fold 2 완료
[no_rounding_no_override] Fold 3 완료
[no_rounding_no_override] Fold 4 완료
[no_rounding_no_override] Fold 5 완료
no_rounding_no_override: CV MAE=0.14999592

[no_rounding_override_010] Fold 1 완료
[no_rounding_override_010] Fold 2 완료
[no_rounding_override_010] Fold 3 완료
[no_rounding_override_010] Fold 4 완료
[no_rounding_override_010] Fold 5 완료
no_rounding_override_010: CV MAE=0.14941229

[pair_k3_no_rounding] Fold 1 완료
[pair_k3_no_rounding] Fold 2 완료
[pair_k3_no_rounding] Fold 3 완료
[pair_k3_no_rounding] Fold 4 완료
[pair_k3_no_rounding] Fold 5 완료
pair_k3_no_rounding: CV MAE=0.15699610

[extra

,name,cv_mae,pair_weight,pair_k,round_decimals,use_override,override_threshold,max_features,min_samples_leaf,criterion,use_feature_copies
0,baseline_original,0.149260,0.24,1,2.0,True,0.2,1.0,1,squared_error,True
1,no_rounding,0.149286,0.24,1,NaN,True,0.2,1.0,1,squared_error,True
2,no_rounding_override_010,0.149412,0.24,1,NaN,True,0.1,1.0,1,squared_error,True
3,no_rounding_no_override,0.149996,0.24,1,NaN,False,0.2,1.0,1,squared_error,True
4,pair_k3_no_rounding,0.156996,0.24,3,NaN,True,0.2,1.0,1,squared_error,True
5,extra_trees_absolute,0.171005,0.24,1,NaN,True,0.2,0.5,2,absolute_error,False
6,extra_trees_mf05_leaf2,0.171730,0.24,1,NaN,True,0.2,0.5,2,squared_error,False


CV 기준 최적 설정: ModelConfig(name='baseline_original', pair_weight=0.24, pair_k=1, round_decimals=2, use_override=True, override_threshold=0.2, max_features=1, min_samples_leaf=1, criterion='squared_error', use_feature_copies=True, include_gender=False)


## Pair weight 미세 탐색

위에서 선택된 모델 구조는 유지하고 `PAIR_WEIGHT`만 미세 조정합니다. 가장 좋은 값을 최종 설정에 반영합니다.


In [6]:
PAIR_WEIGHT_CANDIDATES = [0.16, 0.20, 0.24, 0.28, 0.32, 0.36]
weight_rows = []

for weight in PAIR_WEIGHT_CANDIDATES:
    config = replace(BEST_CONFIG, name=f"{BEST_CONFIG.name}_w{weight:.2f}", pair_weight=weight)
    mae, _ = evaluate_config(config)
    weight_rows.append({"pair_weight": weight, "cv_mae": mae})
    print(f"PAIR_WEIGHT={weight:.2f}: {mae:.8f}\n")

weight_results = pd.DataFrame(weight_rows).sort_values("cv_mae").reset_index(drop=True)
display(weight_results)

BEST_PAIR_WEIGHT = float(weight_results.loc[0, "pair_weight"])
FINAL_CONFIG = replace(
    BEST_CONFIG,
    name=f"final_{BEST_CONFIG.name}_w{BEST_PAIR_WEIGHT:.2f}",
    pair_weight=BEST_PAIR_WEIGHT,
)
print("최종 설정:", FINAL_CONFIG)


[baseline_original_w0.16] Fold 1 완료
[baseline_original_w0.16] Fold 2 완료
[baseline_original_w0.16] Fold 3 완료
[baseline_original_w0.16] Fold 4 완료
[baseline_original_w0.16] Fold 5 완료
PAIR_WEIGHT=0.16: 0.14954667

[baseline_original_w0.20] Fold 1 완료
[baseline_original_w0.20] Fold 2 완료
[baseline_original_w0.20] Fold 3 완료
[baseline_original_w0.20] Fold 4 완료
[baseline_original_w0.20] Fold 5 완료
PAIR_WEIGHT=0.20: 0.14939667

[baseline_original_w0.24] Fold 1 완료
[baseline_original_w0.24] Fold 2 완료
[baseline_original_w0.24] Fold 3 완료
[baseline_original_w0.24] Fold 4 완료
[baseline_original_w0.24] Fold 5 완료
PAIR_WEIGHT=0.24: 0.14926000

[baseline_original_w0.28] Fold 1 완료
[baseline_original_w0.28] Fold 2 완료
[baseline_original_w0.28] Fold 3 완료
[baseline_original_w0.28] Fold 4 완료
[baseline_original_w0.28] Fold 5 완료
PAIR_WEIGHT=0.28: 0.14928667

[baseline_original_w0.32] Fold 1 완료
[baseline_original_w0.32] Fold 2 완료
[baseline_original_w0.32] Fold 3 완료
[baseline_original_w0.32] Fold 4 완료
[baseline_origin

,pair_weight,cv_mae
0,0.36,0.149157
1,0.32,0.149203
2,0.24,0.149260
3,0.28,0.149287
4,0.20,0.149397
5,0.16,0.149547


최종 설정: ModelConfig(name='final_baseline_original_w0.36', pair_weight=0.36, pair_k=1, round_decimals=2, use_override=True, override_threshold=0.2, max_features=1, min_samples_leaf=1, criterion='squared_error', use_feature_copies=True, include_gender=False)


## 최종 제출 파일 생성

CV에서 선택된 `FINAL_CONFIG`로 전체 학습 데이터를 사용하여 예측합니다. 제출 CSV에는 반올림하지 않은 예측값이 그대로 저장될 수 있습니다.


In [7]:
test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)

final_prediction = combined_prediction(
    raw_input,
    test_input,
    y,
    numeric_cols,
    FINAL_CONFIG,
)

output = submission.copy()
output[TARGET] = final_prediction

safe_name = FINAL_CONFIG.name.replace(".", "p")
submit_path = OUTPUT_DIR / f"submit_improved_{safe_name}.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Final config:", FINAL_CONFIG)
print("Prediction range:", final_prediction.min(), final_prediction.max())
print("Prediction mean:", final_prediction.mean())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_improved_final_baseline_original_w0p36.csv
Final config: ModelConfig(name='final_baseline_original_w0.36', pair_weight=0.36, pair_k=1, round_decimals=2, use_override=True, override_threshold=0.2, max_features=1, min_samples_leaf=1, criterion='squared_error', use_feature_copies=True, include_gender=False)
Prediction range: 0.0 1.0
Prediction mean: 0.5076966666666667


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.44
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.52
9,TEST_0009,0.88
